# 🏠 House Price Analysis
**Project 2 — Multi-Domain Data Analysis Portfolio**

**Author:** Vishal Arya | Full Stack Developer & Data Analyst | Servana Tech  
**GitHub:** [VishalAarya89](https://github.com/VishalAarya89)

---

## 📋 Objective
Analyze a residential property dataset to:
- Clean and validate the data
- Compute descriptive statistics
- Identify correlations between features and price
- Compare prices across locations and property types
- Generate visual insights and business recommendations

## 📦 Dataset
`house_prices.csv` — 300 properties with columns: `Property_ID`, `Area`, `Bedrooms`, `Bathrooms`, `Age`, `Location`, `Property_Type`, `Price`

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("house_prices.csv")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

In [ ]:
# Quick structural overview
df.info()

## 3. Data Cleaning

Steps performed:
1. Check for missing values
2. Check for duplicate rows
3. Validate that Price and Area are positive
4. Standardize text columns
5. Engineer a new feature: `Price_per_sqft`

In [ ]:
# Step 1: Missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Step 2: Duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

df = df.drop_duplicates().copy()

In [ ]:
# Step 3: Validate logical constraints (no negative price/area)
invalid = df[(df["Price"] <= 0) | (df["Area"] <= 0)]
print(f"Invalid rows (price/area <= 0): {len(invalid)}")

df = df[(df["Price"] > 0) & (df["Area"] > 0)]

In [ ]:
# Step 4: Standardize text columns
for col in ["Location", "Property_Type"]:
    df[col] = df[col].astype(str).str.strip().str.title()

print("Unique Locations:", df["Location"].unique())
print("Unique Property Types:", df["Property_Type"].unique())

In [ ]:
# Step 5: Feature engineering — Price per square foot
df["Price_per_sqft"] = (df["Price"] / df["Area"]).round(2)

print(f"Final cleaned dataset shape: {df.shape}")
df.head()

## 4. Statistical Analysis

Computing Mean, Median, Mode, Standard Deviation, and Variance for all key numeric features.

In [ ]:
numeric_cols = ["Area", "Bedrooms", "Bathrooms", "Age", "Price", "Price_per_sqft"]

stats_summary = pd.DataFrame({
    "Mean": df[numeric_cols].mean(),
    "Median": df[numeric_cols].median(),
    "Mode": df[numeric_cols].mode().iloc[0],
    "Std Dev": df[numeric_cols].std(),
    "Variance": df[numeric_cols].var(),
    "Min": df[numeric_cols].min(),
    "Max": df[numeric_cols].max(),
}).round(2)

stats_summary

In [ ]:
# Overall describe() for a quick sanity check
df[numeric_cols].describe().round(2)

## 5. Correlation Analysis

Identifying which features have the strongest relationship with `Price`.

In [ ]:
corr_cols = ["Area", "Bedrooms", "Bathrooms", "Age", "Price"]
corr_matrix = df[corr_cols].corr().round(3)
corr_matrix

In [ ]:
# Correlation with Price specifically, sorted
price_corr = corr_matrix["Price"].drop("Price").sort_values(ascending=False)
print("Correlation with Price (ranked):")
price_corr

In [ ]:
# Visualization 1: Correlation Heatmap
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap — Property Features vs Price", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Price Distribution

In [ ]:
# Visualization 2: Price Histogram
plt.figure(figsize=(8, 5))
sns.histplot(df["Price"], bins=30, kde=True, color="#2E5EAA", edgecolor="white")
plt.axvline(df["Price"].mean(), color="#E76F51", linestyle="--", linewidth=2,
            label=f"Mean: ₹{df['Price'].mean():,.0f}")
plt.axvline(df["Price"].median(), color="#43AA8B", linestyle="--", linewidth=2,
            label=f"Median: ₹{df['Price'].median():,.0f}")
plt.title("Price Distribution of Properties", fontsize=12, fontweight="bold")
plt.xlabel("Price (₹)")
plt.ylabel("Number of Properties")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Area vs Price Relationship

In [ ]:
# Visualization 3: Scatter plot — Area vs Price
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=df, x="Area", y="Price", hue="Property_Type",
                 palette=["#2E5EAA", "#5DA9E9", "#F2A65A"], alpha=0.75, s=55, edgecolor="white")

z = np.polyfit(df["Area"], df["Price"], 1)
p = np.poly1d(z)
x_line = np.linspace(df["Area"].min(), df["Area"].max(), 100)
plt.plot(x_line, p(x_line), color="black", linestyle="--", linewidth=1.5, label="Trend Line")

plt.title("Area vs Price Relationship", fontsize=12, fontweight="bold")
plt.xlabel("Area (sq.ft)")
plt.ylabel("Price (₹)")
plt.legend(title="Property Type")
plt.tight_layout()
plt.show()

## 8. Price Distribution by Location

In [ ]:
# Visualization 4: Boxplot by Location
plt.figure(figsize=(8, 5.5))
order = df.groupby("Location")["Price"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Location", y="Price", order=order, hue="Location",
            palette=["#2E5EAA", "#5DA9E9", "#F2A65A"], legend=False)
plt.title("Price Distribution by Location", fontsize=12, fontweight="bold")
plt.xlabel("Location")
plt.ylabel("Price (₹)")
plt.tight_layout()
plt.show()

## 9. Feature Importance (Correlation-Based)

In [ ]:
# Visualization 5: Feature importance bar chart
price_corr_sorted = corr_matrix["Price"].drop("Price").sort_values()

plt.figure(figsize=(8, 5))
colors = ["#E76F51" if v < 0 else "#2E5EAA" for v in price_corr_sorted.values]
bars = plt.barh(price_corr_sorted.index, price_corr_sorted.values, color=colors, edgecolor="white")

for bar, val in zip(bars, price_corr_sorted.values):
    plt.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
             f"{val:.2f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)

plt.title("Feature Importance — Correlation with Price", fontsize=12, fontweight="bold")
plt.xlabel("Correlation Coefficient")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. Location-Based Analysis

In [ ]:
location_summary = df.groupby("Location").agg(
    Avg_Price=("Price", "mean"),
    Median_Price=("Price", "median"),
    Avg_Area=("Area", "mean"),
    Avg_Price_per_sqft=("Price_per_sqft", "mean"),
    Property_Count=("Property_ID", "count"),
).round(2).sort_values("Avg_Price", ascending=False)

location_summary

## 11. Property Type Analysis

In [ ]:
type_summary = df.groupby("Property_Type").agg(
    Avg_Price=("Price", "mean"),
    Median_Price=("Price", "median"),
    Avg_Area=("Area", "mean"),
    Property_Count=("Property_ID", "count"),
).round(2).sort_values("Avg_Price", ascending=False)

type_summary

In [ ]:
# Visualization 6: Property Type Distribution (Pie Chart)
counts = df["Property_Type"].value_counts()

plt.figure(figsize=(6.5, 6.5))
plt.pie(counts.values, labels=counts.index, autopct="%1.1f%%",
        colors=["#2E5EAA", "#5DA9E9", "#F2A65A"], startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5}, textprops={"fontsize": 10})
plt.title("Property Type Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 12. Bedroom Impact Analysis

In [ ]:
bedroom_summary = df.groupby("Bedrooms").agg(
    Avg_Price=("Price", "mean"),
    Avg_Area=("Area", "mean"),
    Count=("Property_ID", "count"),
).round(2)

bedroom_summary

In [ ]:
# Visualization 7: Average Price by Bedroom Count
summary = df.groupby("Bedrooms")["Price"].mean().round(0)

plt.figure(figsize=(7, 5))
plt.bar(summary.index.astype(str), summary.values, color="#2E5EAA", edgecolor="white")
for i, v in enumerate(summary.values):
    plt.text(i, v + summary.values.max()*0.01, f"₹{v:,.0f}", ha="center", fontsize=9)
plt.title("Average Price by Bedroom Count", fontsize=12, fontweight="bold")
plt.xlabel("Number of Bedrooms")
plt.ylabel("Average Price (₹)")
plt.tight_layout()
plt.show()

## 13. Property Age vs Price

In [ ]:
# Visualization 8: Age vs Price scatter
plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=df, x="Age", y="Price", color="#2E5EAA", alpha=0.6, s=50)
z = np.polyfit(df["Age"], df["Price"], 1)
p = np.poly1d(z)
x_line = np.linspace(df["Age"].min(), df["Age"].max(), 100)
plt.plot(x_line, p(x_line), color="#E76F51", linestyle="--", linewidth=2, label="Trend Line")
plt.title("Property Age vs Price", fontsize=12, fontweight="bold")
plt.xlabel("Age (years)")
plt.ylabel("Price (₹)")
plt.legend()
plt.tight_layout()
plt.show()

## 14. Key Business Insights

Based on the complete analysis above:

1. **Area is the dominant price driver** — correlation of **0.80** with Price, far stronger than any other feature.
2. **Bedrooms show a moderate positive relationship** (corr = 0.20), while **Bathrooms (-0.03) and Age (-0.13)** have minimal/negative impact.
3. **City Center** is the most valuable location (avg ₹33.1M), nearly **2x** the average price of **Rural** properties (avg ₹16.5M).
4. **Apartments** have the highest average price among property types, despite Houses having the most listings.
5. **5-bedroom homes** command the highest average price (₹29.4M), but the relationship isn't perfectly linear — 3-bedroom homes are actually cheaper on average than 2-bedroom homes, suggesting other factors (like location) are at play.

## 15. Recommendations

**For Investors:**
- Focus on **City Center** properties for the highest absolute returns, or **Rural** properties for the best price-per-sqft entry point.
- Prioritize **Area** over bedroom/bathroom count when evaluating investment potential — it's the strongest, most reliable price predictor.

**For Sellers (Pricing Strategy):**
- Price primarily based on **square footage**, not just room counts.
- Account for a modest price discount as property **Age** increases.

**For Buyers:**
- Compare **Price per sq.ft** across locations rather than absolute price — Rural areas offer significantly more space per rupee spent.

---
*Notebook generated as part of a Multi-Domain Data Analysis Portfolio by Vishal Arya, Servana Tech.*